In [10]:
import pandas as pd
import numpy as np
import pyodbc
import warnings

pd.set_option('display.max_columns', 100)
pd.set_option('display.min_rows', 50)

def run_sql(query):
    with pyodbc.connect("DSN=Redshift_prod_new") as conn:
        warnings.filterwarnings("ignore", category=UserWarning)
        df = pd.read_sql_query(sql=query, con=conn)
        warnings.filterwarnings("default", category=UserWarning)
    return df

In [11]:
ste_t1_query = """
SELECT *
FROM sandbox.t_1
WHERE effective_date >= '2025-12-01'
  AND roa_dealer_group = 'SFS'
"""

ste_t1_raw = run_sql(ste_t1_query)
ste_t1_raw['effective_date'] = pd.to_datetime(ste_t1_raw['effective_date'])
print(f"Raw STE T1 rows: {len(ste_t1_raw)}")
print(f"Date range: {ste_t1_raw['effective_date'].min().date()} to {ste_t1_raw['effective_date'].max().date()}")
ste_t1_raw.head()

Raw STE T1 rows: 19
Date range: 2025-12-01 to 2026-05-18


,roa_dealer_group,rowid,lookup,notes,datetime_inserted,effective_date,halfbaked_date,mroa,orig_yield,cnl_effect,var_cost,cost_of_debt,run_rate_nb,model_score,model_name,wal,wal_cfc,wal_adj_factor,apr,apr_realization_factor,discount,fee_income,pmt_proc_income,addl_income,cnl,gross_loss,recovery,cnl_adj_factor,gross_loss_cfc,gross_loss_adj_factor,recovery_cfc,recovery_adj_factor,debt_int_rate,securitization_cost,debt_adv_rate,orig_cost,serv_cost,addl_cost,days_till_half_baked,sort,latest_row_flag,loss_meeting_flag,pricing_change_flag,is_current_flag,elasticity,nb_ratio,version_start_datetime,version_end_datetime,ticket_no,addl_cost_of_debt,tier,fixed_cost,subdebt_expense,vroa,ragu_score,baseline_ltv,ltv,baseline_apr,ragu_gross_loss_adjustment,ragu_recovery_adjustment,ragu_ltv_adjustment,ragu_market_expectation_adjustment,ragu_apr_adjustment
0,SFS,29,SFS29,Restated Loss Estimates for what we are bookin...,2026-01-08 14:17:03,2026-01-01,None,0.051797,0.234250,0.122271,0.015,0.045182,500.0,0.0,,2.27,2.27,1.0,0.235,0.905,0.029,0.003,0.0041,0.0017,0.277556,0.484724,0.207169,1.050551,0.4614,1.0,0.1972,1.0,0.0525,0.0026,0.82,0.005675,0.0125,0,2,10,0,1,0,1,0.01,1.0,2026-01-08 14:17:03,9999-01-01,NaN,0.0,None,0.005,0.0027,0.044097,None,None,None,None,None,None,None,None,None
1,SFS,33,SFS33,Cost of Funds Update (5.60%),2026-03-19 00:00:00,2026-03-19,None,0.056485,0.234250,0.114714,0.015,0.048052,500.0,0.0,,2.27,2.27,1.0,0.235,0.905,0.029,0.003,0.0041,0.0017,0.260400,0.454764,0.194364,0.985617,0.4614,1.0,0.1972,1.0,0.0560,0.0026,0.82,0.005675,0.0125,0,2,10,0,0,0,1,0.01,1.0,2026-03-19 19:56:46,9999-01-01,NaN,0.0,None,0.005,0.0027,0.048785,None,None,None,None,None,None,None,None,None
2,SFS,34,SFS34,Cost of Funds Update (5.85%),2026-03-30 00:00:00,2026-03-30,None,0.054435,0.234250,0.114714,0.015,0.050102,500.0,0.0,,2.27,2.27,1.0,0.235,0.905,0.029,0.003,0.0041,0.0017,0.260400,0.454764,0.194364,0.985617,0.4614,1.0,0.1972,1.0,0.0585,0.0026,0.82,0.005675,0.0125,0,2,10,0,0,0,1,0.01,1.0,2026-03-30 15:45:25,9999-01-01,NaN,0.0,None,0.005,0.0027,0.046735,None,None,None,None,None,None,None,None,None
3,SFS,36,SFS36,Volume cuts (from frontier meeting 3/26),2026-03-31 00:00:00,2026-03-31,2026-04-02,0.056885,0.235572,0.113585,0.015,0.050102,760.0,0.0,,2.27,2.27,1.0,0.235,0.905,0.032,0.003,0.0041,0.0017,0.257837,0.452201,0.194364,0.985617,0.4588,1.0,0.1972,1.0,0.0585,0.0026,0.82,0.005675,0.0125,0,2,10,0,0,1,1,0.01,1.0,2026-03-31 00:00:00,9999-01-01,ORGN-42106/42108/42110,0.0,None,0.005,0.0027,0.049185,None,None,None,None,None,None,None,None,None
4,SFS,41,SFS41,Cost of Funds Update (5.7%),2026-05-05 00:00:00,2026-05-04,None,0.045233,0.226153,0.117048,0.015,0.048872,710.0,0.0,,2.27,2.27,1.0,0.228,0.905,0.025,0.003,0.0041,0.0017,0.265700,0.468001,0.202301,1.025869,0.4562,1.0,0.1972,1.0,0.0570,0.0026,0.82,0.005675,0.0125,0,2,10,0,0,0,1,0.01,1.0,2026-05-05 18:34:12,9999-01-01,NaN,0.0,None,0.005,0.0027,0.037533,None,None,None,None,None,None,None,None,None


In [12]:
# Deduplicate: one row per month (latest effective_date, then highest rowid for ties)
ste_t1_raw['year_month'] = ste_t1_raw['effective_date'].dt.to_period('M')

ste_t1_deduped = (ste_t1_raw
    .sort_values(['effective_date', 'rowid'], ascending=[False, False])
    .drop_duplicates(subset='year_month', keep='first')
    .sort_values('effective_date')
    .reset_index(drop=True))

print(f"Deduplicated rows: {len(ste_t1_deduped)} (one per month)")
print(f"Months: {ste_t1_deduped['year_month'].tolist()}")

# Select standard T1 columns
t1_cols = ['roa_dealer_group', 'effective_date', 'model_score', 'discount', 'apr',
           'ragu_score', 'cnl', 'wal', 'apr_realization_factor', 'fee_income',
           'pmt_proc_income', 'addl_income', 'var_cost', 'cost_of_debt']

ste_t1_clean = ste_t1_deduped[t1_cols].copy()

# Compute orig_yield (same formula as other LOBs)
ste_t1_clean['orig_yield'] = (ste_t1_clean['apr'] * ste_t1_clean['apr_realization_factor']
                              + ste_t1_clean['discount'] / ste_t1_clean['wal']
                              + ste_t1_clean['fee_income']
                              + ste_t1_clean['pmt_proc_income']
                              + ste_t1_clean['addl_income'])

ste_t1_clean = ste_t1_clean.sort_values('effective_date').reset_index(drop=True)

# Print the SFS27 row (effective_date = 2025-12-01) -- use for loans originated before December
sfs27 = ste_t1_raw[ste_t1_raw['lookup'] == 'SFS27'].iloc[0]
print("=" * 70)
print("  SFS27 (effective_date = 2025-12-01) -- use for pre-December loans")
print("=" * 70)
print(f"  effective_date: {sfs27['effective_date'].date()}")
print(f"  CNL:            {sfs27['cnl']}")
print(f"  WAL:            {sfs27['wal']}")
print(f"  APR:            {sfs27['apr']}")
print(f"  discount:       {sfs27['discount']}")
print(f"  var_cost:       {sfs27['var_cost']}")
print(f"  cost_of_debt:   {sfs27['cost_of_debt']}")
print("=" * 70)
print()

ste_t1_clean

Deduplicated rows: 6 (one per month)
Months: [Period('2025-12', 'M'), Period('2026-01', 'M'), Period('2026-02', 'M'), Period('2026-03', 'M'), Period('2026-04', 'M'), Period('2026-05', 'M')]
  SFS27 (effective_date = 2025-12-01) -- use for pre-December loans
  effective_date: 2025-12-01
  CNL:            0.280890274896
  WAL:            2.27
  APR:            0.235
  discount:       0.029
  var_cost:       0.015
  cost_of_debt:   0.045182



,roa_dealer_group,effective_date,model_score,discount,apr,ragu_score,cnl,wal,apr_realization_factor,fee_income,pmt_proc_income,addl_income,var_cost,cost_of_debt,orig_yield
0,SFS,2025-12-18,0.0,0.029,0.235,None,0.262400,2.27,0.905,0.003,0.0041,0.0017,0.015,0.045182,0.234250
1,SFS,2026-01-01,0.0,0.029,0.235,None,0.277556,2.27,0.905,0.003,0.0041,0.0017,0.015,0.045182,0.234250
2,SFS,2026-02-16,0.0,0.029,0.235,None,0.277556,2.27,0.905,0.003,0.0041,0.0017,0.015,0.043952,0.234250
3,SFS,2026-03-31,0.0,0.032,0.235,None,0.257837,2.27,0.905,0.003,0.0041,0.0017,0.015,0.050102,0.235572
4,SFS,2026-04-22,0.0,0.025,0.228,None,0.265700,2.27,0.905,0.003,0.0041,0.0017,0.015,0.047232,0.226153
5,SFS,2026-05-18,0.0,0.018,0.228,None,0.241079,2.27,0.920,0.003,0.0041,0.0017,0.014,0.050512,0.226490


In [13]:
output_file = 'ragu_profit_analysis.xlsx'

with pd.ExcelWriter(output_file, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
    ste_t1_clean.to_excel(writer, sheet_name='T1 STE', index=False)

print(f"Exported 'T1 STE' sheet to {output_file} ({len(ste_t1_clean)} rows)")

Exported 'T1 STE' sheet to ragu_profit_analysis.xlsx (6 rows)


In [14]:
# Load STE contract-level data and join with loan_random_numbers for hurdle split
with open('ste_query', 'r') as f:
    ste_contract_query = f.read()

ste_df = run_sql(ste_contract_query)
ste_df['application_received_dtm'] = pd.to_datetime(ste_df['application_received_dtm'])

# Filter to 2025+
ste_df = ste_df[ste_df['application_received_dtm'] >= '2025-01-01'].copy()
print(f"STE contracts since 2025-01-01: {len(ste_df)}")

# Join loan_random_numbers via los_deal_current_fact (data_source_id=101 = STE)
lrn_query = """
SELECT ldcf.account_number, lrn.maxltv
FROM los_deal_current_fact ldcf
LEFT JOIN sandbox.loan_random_numbers lrn
    ON lrn.loan_id = ldcf.loan_id
WHERE ldcf.data_source_id = 101
  AND ldcf.application_received_dtm >= '2025-01-01'
  AND aspect = 'CONTRACT'
"""
lrn_df = run_sql(lrn_query)

ste_df = ste_df.merge(
    lrn_df[['account_number', 'maxltv']].drop_duplicates(subset='account_number', keep='first'),
    on='account_number', how='left'
)

print(f"After join with loan_random_numbers: {len(ste_df)} rows")
print(f"maxltv coverage: {ste_df['maxltv'].notna().sum()} / {len(ste_df)} ({ste_df['maxltv'].notna().mean():.1%})")
ste_df[['account_number', 'application_received_dtm', 'con_risk_model_score', 'discount', 'discount_percent', 'con_apr', 'con_amount_financed_back', 'maxltv']].head(10)

STE contracts since 2025-01-01: 16478
After join with loan_random_numbers: 16478 rows
maxltv coverage: 16113 / 16478 (97.8%)


,account_number,application_received_dtm,con_risk_model_score,discount,discount_percent,con_apr,con_amount_financed_back,maxltv
0,9.012514e+10,2025-11-27 00:42:59,129.0,1737.63,0.066738,0.2490,26036.65,104.0
1,9.012510e+10,2025-10-11 19:10:02,135.0,1848.00,0.030333,0.1800,60923.50,675.0
2,9.012514e+10,2025-12-19 16:35:03,126.0,3503.07,0.118777,0.2490,29492.79,930.0
3,9.012513e+10,2025-11-18 21:03:37,NaN,NaN,NaN,0.2698,22287.90,83.0
4,9.012516e+10,2026-01-05 22:39:58,142.0,1148.00,0.016394,0.1673,70027.72,793.0
5,9.012516e+10,2026-01-17 23:26:13,132.0,2016.09,0.110820,0.2699,18192.41,848.0
6,9.012514e+10,2025-12-09 21:46:40,137.0,-503.71,-0.019932,0.2799,25270.95,532.0
7,9.012513e+10,2025-11-04 22:36:06,130.0,-2.00,-0.000052,0.2799,38432.48,969.0
8,9.012514e+10,2025-11-21 18:49:30,134.0,-455.78,-0.017889,0.2698,25477.95,989.0
9,9.012515e+10,2025-12-30 00:54:17,132.0,-2.00,-0.000053,0.2698,37629.72,398.0


In [15]:
# Assign hurdle based on maxltv (same logic as ragu_by_hurdle_pool.sql for data_source_id=101)
ste_df['hurdle'] = np.where(
    ste_df['maxltv'].fillna(0) < 500,
    '2) Higher',
    '1) Lower'
)

ste_df['app_month'] = ste_df['application_received_dtm'].dt.to_period('M').dt.to_timestamp()

# Aggregate by month + hurdle
def weighted_avg_notnull(values, weights):
    mask = values.notna()
    if weights[mask].sum() == 0:
        return np.nan
    return (values[mask] * weights[mask]).sum() / weights[mask].sum()

ste_agg = ste_df.groupby(['app_month', 'hurdle']).apply(
    lambda g: pd.Series({
        'loan_count': g['account_number'].nunique(),
        'weighted_model_score': weighted_avg_notnull(g['con_risk_model_score'], g['con_amount_financed_back']),
        'weighted_discount_pct': g['discount'].sum() / g['con_amount_financed_back'].sum() if g['con_amount_financed_back'].sum() != 0 else np.nan,
        'weighted_apr': (g['con_apr'] * g['con_amount_financed_back']).sum() / g['con_amount_financed_back'].sum() if g['con_amount_financed_back'].sum() != 0 else np.nan,
    })
).reset_index()

ste_agg = ste_agg.sort_values(['app_month', 'hurdle']).reset_index(drop=True)

print(f"STE aggregated: {len(ste_agg)} rows")
print(f"Months: {ste_agg['app_month'].dt.strftime('%Y-%m').unique().tolist()}")
print()
ste_agg

STE aggregated: 16 rows
Months: ['2025-10', '2025-11', '2025-12', '2026-01', '2026-02', '2026-03', '2026-04', '2026-05']



,app_month,hurdle,loan_count,weighted_model_score,weighted_discount_pct,weighted_apr
0,2025-10-01,1) Lower,868.0,131.130006,0.023224,0.234246
1,2025-10-01,2) Higher,866.0,131.157369,0.020115,0.235532
2,2025-11-01,1) Lower,866.0,131.557892,0.026380,0.233469
3,2025-11-01,2) Higher,737.0,131.730451,0.025844,0.234008
4,2025-12-01,1) Lower,736.0,132.043993,0.026775,0.233375
5,2025-12-01,2) Higher,645.0,131.797046,0.027814,0.236692
6,2026-01-01,1) Lower,747.0,131.985764,0.026517,0.235625
7,2026-01-01,2) Higher,681.0,132.972369,0.024010,0.235512
8,2026-02-01,1) Lower,1479.0,133.551307,0.024378,0.231872
9,2026-02-01,2) Higher,1355.0,133.919746,0.025868,0.230958


In [16]:
import openpyxl

ragu_file = 'ste_hurdle_ragu.xlsx'
wb_ragu = openpyxl.load_workbook(ragu_file, read_only=True)
ws = wb_ragu['Data Tables (M)']

ragu_rows = []
current_hurdle = None
vintage_cols = []

for row in ws.iter_rows(min_row=1, values_only=True):
    first_cell = row[0]
    if first_cell and str(first_cell).startswith('STE - '):
        current_hurdle = first_cell.replace('STE - ', '')
        vintage_cols = [c for c in row[1:] if c is not None]
    elif first_cell == 'RAGU Score' and current_hurdle:
        scores = row[1:len(vintage_cols) + 1]
        for vintage_str, score in zip(vintage_cols, scores):
            if score is not None:
                year, mpart = vintage_str.split(' M')
                app_month = pd.Timestamp(f"{year}-{mpart}-01")
                ragu_rows.append({'hurdle': current_hurdle, 'app_month': app_month, 'ragu_score': score})

wb_ragu.close()

ragu_lookup = pd.DataFrame(ragu_rows)
ste_agg = ste_agg.merge(ragu_lookup, on=['hurdle', 'app_month'], how='left')
print(f"RAGU scores merged: {ste_agg['ragu_score'].notna().sum()} / {len(ste_agg)} rows populated")
ste_agg

RAGU scores merged: 16 / 16 rows populated


,app_month,hurdle,loan_count,weighted_model_score,weighted_discount_pct,weighted_apr,ragu_score
0,2025-10-01,1) Lower,868.0,131.130006,0.023224,0.234246,139.405036
1,2025-10-01,2) Higher,866.0,131.157369,0.020115,0.235532,138.704639
2,2025-11-01,1) Lower,866.0,131.557892,0.026380,0.233469,138.454412
3,2025-11-01,2) Higher,737.0,131.730451,0.025844,0.234008,138.849280
4,2025-12-01,1) Lower,736.0,132.043993,0.026775,0.233375,138.015832
5,2025-12-01,2) Higher,645.0,131.797046,0.027814,0.236692,138.414314
6,2026-01-01,1) Lower,747.0,131.985764,0.026517,0.235625,138.860533
7,2026-01-01,2) Higher,681.0,132.972369,0.024010,0.235512,140.391251
8,2026-02-01,1) Lower,1479.0,133.551307,0.024378,0.231872,139.580570
9,2026-02-01,2) Higher,1355.0,133.919746,0.025868,0.230958,140.711974


In [17]:
# Build T-1 lookup: shift effective_date forward by 1 month
t1_for_t0 = ste_t1_clean[['effective_date', 'cnl', 'wal', 'apr_realization_factor',
                            'var_cost', 'cost_of_debt',
                            'fee_income', 'pmt_proc_income', 'addl_income']].copy()
t1_for_t0['t1_ragu_score'] = 143.0
t1_for_t0['t1_cnl'] = t1_for_t0['cnl']
t1_for_t0['effective_date'] = pd.to_datetime(t1_for_t0['effective_date'])
t1_for_t0['app_month'] = t1_for_t0['effective_date'] + pd.offsets.MonthBegin(1)

# For months <= December 2025, use the earliest T-1 row (December 2025)
dec_row = t1_for_t0.sort_values('effective_date').iloc[0]
early_months = ste_agg.loc[ste_agg['app_month'] <= '2025-12-01', 'app_month'].unique()
if len(early_months) > 0:
    early_rows = pd.DataFrame([dec_row] * len(early_months))
    early_rows['app_month'] = early_months
    t1_for_t0 = pd.concat([t1_for_t0, early_rows], ignore_index=True).drop_duplicates(subset='app_month', keep='last')

# Merge T-1 parameters onto ste_agg (same T-1 for both hurdles within a month)
ste_agg = ste_agg.merge(
    t1_for_t0[['app_month', 't1_ragu_score', 't1_cnl', 'apr_realization_factor',
               'wal', 'var_cost', 'cost_of_debt',
               'fee_income', 'pmt_proc_income', 'addl_income']],
    on='app_month', how='left'
)

# Compute orig_yield (actual T0 apr/discount + T-1 structural assumptions)
ste_agg['orig_yield'] = (ste_agg['weighted_apr'] * ste_agg['apr_realization_factor']
                         + ste_agg['weighted_discount_pct'] / ste_agg['wal']
                         + ste_agg['fee_income']
                         + ste_agg['pmt_proc_income']
                         + ste_agg['addl_income'])

# Compute t0_cnl: adjust T-1 CNL by RAGU delta
ste_agg['t0_cnl'] = ste_agg['t1_cnl'] - (ste_agg['ragu_score'] - ste_agg['t1_ragu_score']) * 0.0065
ste_agg['cnl_effect'] = ste_agg['t0_cnl'] / ste_agg['wal']

# Compute mROA
ste_agg['mroa'] = (ste_agg['orig_yield']
                   - ste_agg['cnl_effect']
                   - ste_agg['var_cost']
                   - ste_agg['cost_of_debt'])

# Drop intermediate T-1 columns
ste_agg = ste_agg.drop(columns=['t1_ragu_score', 't1_cnl', 'apr_realization_factor',
                                 'fee_income', 'pmt_proc_income', 'addl_income'])

# Compute mroa_gap, mmroa, vol_change
lower = ste_agg[ste_agg['hurdle'] == '1) Lower'].set_index('app_month')
higher = ste_agg[ste_agg['hurdle'] == '2) Higher'].set_index('app_month')

common_months = lower.index.intersection(higher.index)
mroa_gap = higher.loc[common_months, 'mroa'].values - lower.loc[common_months, 'mroa'].values
mmroa = ((higher.loc[common_months, 'mroa'] * higher.loc[common_months, 'loan_count']
          - lower.loc[common_months, 'mroa'] * lower.loc[common_months, 'loan_count'])
         / (higher.loc[common_months, 'loan_count'] - lower.loc[common_months, 'loan_count']))
vol_change = higher.loc[common_months, 'loan_count'] / lower.loc[common_months, 'loan_count'] - 1

ste_agg['mroa_gap'] = np.nan
ste_agg['mmroa'] = np.nan
ste_agg['vol_change'] = np.nan
higher_mask = ste_agg['hurdle'] == '2) Higher'
ste_agg.loc[higher_mask & ste_agg['app_month'].isin(common_months), 'mroa_gap'] = mroa_gap
ste_agg.loc[higher_mask & ste_agg['app_month'].isin(common_months), 'mmroa'] = mmroa.values
ste_agg.loc[higher_mask & ste_agg['app_month'].isin(common_months), 'vol_change'] = vol_change.values

print(f"mROA computed: {ste_agg['mroa'].notna().sum()} / {len(ste_agg)} rows")
print(f"mmROA computed: {ste_agg['mmroa'].notna().sum()} rows (Higher hurdle only)")
ste_agg

mROA computed: 16 / 16 rows
mmROA computed: 8 rows (Higher hurdle only)


,app_month,hurdle,loan_count,weighted_model_score,weighted_discount_pct,weighted_apr,ragu_score,wal,var_cost,cost_of_debt,orig_yield,t0_cnl,cnl_effect,mroa,mroa_gap,mmroa,vol_change
0,2025-10-01,1) Lower,868.0,131.130006,0.023224,0.234246,139.405036,2.27,0.015,0.045182,0.231023,0.285767,0.125889,0.044952,NaN,NaN,NaN
1,2025-10-01,2) Higher,866.0,131.157369,0.020115,0.235532,138.704639,2.27,0.015,0.045182,0.230818,0.290320,0.127894,0.042742,-0.002210,1.002088,-0.002304
2,2025-11-01,1) Lower,866.0,131.557892,0.026380,0.233469,138.454412,2.27,0.015,0.045182,0.231711,0.291946,0.128611,0.042918,NaN,NaN,NaN
3,2025-11-01,2) Higher,737.0,131.730451,0.025844,0.234008,138.849280,2.27,0.015,0.045182,0.231962,0.289380,0.127480,0.044300,0.001382,0.035022,-0.148961
4,2025-12-01,1) Lower,736.0,132.043993,0.026775,0.233375,138.015832,2.27,0.015,0.045182,0.231800,0.294797,0.129867,0.041751,NaN,NaN,NaN
5,2025-12-01,2) Higher,645.0,131.797046,0.027814,0.236692,138.414314,2.27,0.015,0.045182,0.235259,0.292207,0.128726,0.046352,0.004600,0.009144,-0.123641
6,2026-01-01,1) Lower,747.0,131.985764,0.026517,0.235625,138.860533,2.27,0.015,0.045182,0.233722,0.289307,0.127448,0.046092,NaN,NaN,NaN
7,2026-01-01,2) Higher,681.0,132.972369,0.024010,0.235512,140.391251,2.27,0.015,0.045182,0.232515,0.279357,0.123065,0.049269,0.003176,0.013318,-0.088353
8,2026-02-01,1) Lower,1479.0,133.551307,0.024378,0.231872,139.580570,2.27,0.015,0.045182,0.229384,0.299782,0.132062,0.037139,NaN,NaN,NaN
9,2026-02-01,2) Higher,1355.0,133.919746,0.025868,0.230958,140.711974,2.27,0.015,0.045182,0.229212,0.292428,0.128823,0.040207,0.003068,0.003610,-0.083840


In [18]:
# Export STE-mmroa sheet to the main workbook
output_file = 'ragu_profit_analysis.xlsx'

with pd.ExcelWriter(output_file, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
    ste_agg.to_excel(writer, sheet_name='STE-mmroa', index=False)

print(f"Exported 'STE-mmroa' sheet to {output_file} ({len(ste_agg)} rows)")

Exported 'STE-mmroa' sheet to ragu_profit_analysis.xlsx (16 rows)


In [19]:
TODAY = pd.Timestamp.now().normalize()

with open('ste_conversion.sql', 'r') as f:
    ste_conv_query = f.read()

conv_df = run_sql(ste_conv_query)
conv_df['app_month'] = pd.to_datetime(conv_df['app_month'])
conv_df['month'] = conv_df['app_month'].dt.month

print(f"STE conversion data: {len(conv_df)} months")
print(f"Date range: {conv_df['app_month'].min().date()} to {conv_df['app_month'].max().date()}")
print()
conv_df

STE conversion data: 9 months
Date range: 2025-10-01 to 2026-06-01



,app_month,total_apps,straight_approved,booked_straight,straight_approval_rate,conversion_rate,month
0,2025-10-01,47550,20907,1656,0.439685,0.079208,10
1,2025-11-01,51236,21291,1555,0.415548,0.073036,11
2,2025-12-01,50690,19824,1344,0.391083,0.067797,12
3,2026-01-01,52390,21436,1405,0.409162,0.065544,1
4,2026-02-01,64802,31514,2794,0.486312,0.088659,2
5,2026-03-01,78869,38429,3543,0.487251,0.092196,3
6,2026-04-01,66010,29642,2496,0.449053,0.084205,4
7,2026-05-01,63709,26834,1343,0.421196,0.050048,5
8,2026-06-01,1334,503,0,0.377061,0.000000,6


In [20]:
sa_overall_avg = conv_df['straight_approval_rate'].mean()
conv_overall_avg = conv_df['conversion_rate'].mean()

dec_mask = conv_df['month'] == 12
feb_mar_mask = (conv_df['month'] == 2) | (conv_df['month'] == 3)

sa_dec_avg = conv_df.loc[dec_mask, 'straight_approval_rate'].mean()
sa_feb_mar_avg = conv_df.loc[feb_mar_mask, 'straight_approval_rate'].mean()
conv_dec_avg = conv_df.loc[dec_mask, 'conversion_rate'].mean()
conv_feb_mar_avg = conv_df.loc[feb_mar_mask, 'conversion_rate'].mean()

print("=" * 70)
print("  STE STRAIGHT APPROVAL RATE")
print("=" * 70)
print(f"Overall avg: {sa_overall_avg:.4f}")
if not np.isnan(sa_dec_avg):
    print(f"December avg: {sa_dec_avg:.4f} ({sa_dec_avg / sa_overall_avg:.3f}x vs overall)")
if not np.isnan(sa_feb_mar_avg):
    print(f"Feb-Mar avg: {sa_feb_mar_avg:.4f} ({sa_feb_mar_avg / sa_overall_avg:.3f}x vs overall)")

print()
print("=" * 70)
print("  STE CONVERSION RATE (booked / straight approved)")
print("=" * 70)
print(f"Overall avg: {conv_overall_avg:.4f}")
if not np.isnan(conv_dec_avg):
    print(f"December avg: {conv_dec_avg:.4f} ({conv_dec_avg / conv_overall_avg:.3f}x vs overall)")
if not np.isnan(conv_feb_mar_avg):
    print(f"Feb-Mar avg: {conv_feb_mar_avg:.4f} ({conv_feb_mar_avg / conv_overall_avg:.3f}x vs overall)")

SEASONALITY_FACTOR = 1.15

def in_feb_mar(app_month):
    return app_month.month in (2, 3)

def seasonally_adjusted_avg(df_slice, col):
    adjusted = df_slice[col].copy()
    seasonal_mask = df_slice['app_month'].apply(in_feb_mar)
    adjusted.loc[seasonal_mask] = adjusted.loc[seasonal_mask] / SEASONALITY_FACTOR
    return adjusted.mean(), seasonal_mask.sum()

last_1m_start = TODAY - pd.DateOffset(months=1)
last_3m_start = TODAY - pd.DateOffset(months=3)
last_6m_start = TODAY - pd.DateOffset(months=6)

mask_1m = (conv_df['app_month'] >= last_1m_start) & (conv_df['app_month'] < TODAY)
mask_3m = (conv_df['app_month'] >= last_3m_start) & (conv_df['app_month'] < TODAY)
mask_3to6m = (conv_df['app_month'] >= last_6m_start) & (conv_df['app_month'] < last_3m_start)

for col, label in [('straight_approval_rate', 'Straight Approval Rate'),
                    ('conversion_rate', 'Conversion Rate')]:
    print(f"\n--- {label} Anchored to {TODAY.date()} (Feb-Mar adj: {SEASONALITY_FACTOR}) ---")
    for name, mask in [('Last 1 month', mask_1m),
                       ('Last 3 months', mask_3m),
                       ('3-6 months ago', mask_3to6m)]:
        raw = conv_df.loc[mask, col].mean()
        adj, adj_n = seasonally_adjusted_avg(conv_df.loc[mask], col)
        print(f"  {name}: Raw={raw:.4f}  | Adj={adj:.4f}  ({adj_n} months seasonally adjusted)")

  STE STRAIGHT APPROVAL RATE
Overall avg: 0.4307
December avg: 0.3911 (0.908x vs overall)
Feb-Mar avg: 0.4868 (1.130x vs overall)

  STE CONVERSION RATE (booked / straight approved)
Overall avg: 0.0667
December avg: 0.0678 (1.016x vs overall)
Feb-Mar avg: 0.0904 (1.355x vs overall)

--- Straight Approval Rate Anchored to 2026-06-01 (Feb-Mar adj: 1.15) ---
  Last 1 month: Raw=0.4212  | Adj=0.4212  (0 months seasonally adjusted)
  Last 3 months: Raw=0.4525  | Adj=0.4313  (1 months seasonally adjusted)
  3-6 months ago: Raw=0.4289  | Adj=0.4077  (1 months seasonally adjusted)

--- Conversion Rate Anchored to 2026-06-01 (Feb-Mar adj: 1.15) ---
  Last 1 month: Raw=0.0500  | Adj=0.0500  (0 months seasonally adjusted)
  Last 3 months: Raw=0.0755  | Adj=0.0715  (1 months seasonally adjusted)
  3-6 months ago: Raw=0.0740  | Adj=0.0701  (1 months seasonally adjusted)


In [21]:
output_file = 'ragu_profit_analysis.xlsx'

export_df = conv_df[['app_month', 'total_apps', 'straight_approved', 'booked_straight',
                      'straight_approval_rate', 'conversion_rate']].copy()

with pd.ExcelWriter(output_file, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
    export_df.to_excel(writer, sheet_name='STE-conversion', index=False)

print(f"Exported 'STE-conversion' sheet to {output_file} ({len(export_df)} rows)")

Exported 'STE-conversion' sheet to ragu_profit_analysis.xlsx (9 rows)
